In [10]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("ci_outputs/edges_ci_sample.csv")
assert DATA_PATH.exists(), "❌ Fichier CI introuvable"

df = pd.read_csv(DATA_PATH)

print("📦 Dataset CI chargé")
print("Lignes :", len(df))
print("Colonnes :", list(df.columns))


📦 Dataset CI chargé
Lignes : 74
Colonnes : ['city', 'trip_id', 'route_id', 'from_stop_id', 'to_stop_id', 'from_lat', 'from_lon', 'to_lat', 'to_lon', 'travel_time_sec', 'delta_lat', 'delta_lon']


In [ ]:
#Tests SCHÉMA
expected_columns = [
    "city","trip_id","route_id",
    "from_stop_id","to_stop_id",
    "from_lat","from_lon",
    "to_lat","to_lon",
    "travel_time_sec",
    "delta_lat","delta_lon"
]

missing = [c for c in expected_columns if c not in df.columns]
assert not missing, f"Colonnes manquantes : {missing}"

print("✅ TEST SCHÉMA (colonnes)")


✅ TEST SCHÉMA (colonnes)


In [12]:
#Tests QUALITÉ des données
key_cols = ["trip_id","from_stop_id","to_stop_id"]

null_keys = df[key_cols].isnull().any(axis=1).sum()
assert null_keys == 0, f"Clés nulles détectées : {null_keys}"

print("✅ TEST QUALITÉ (clés non nulles)")


✅ TEST QUALITÉ (clés non nulles)


In [13]:
#Tests LOGIQUE MÉTIER (copie fidèle du Spark)
#Self-loops
self_loops = df[df["from_stop_id"] == df["to_stop_id"]]
normal_edges = df[df["from_stop_id"] != df["to_stop_id"]]

print("Self-loops :", len(self_loops))
print("Normal edges :", len(normal_edges))


Self-loops : 16
Normal edges : 58


In [14]:
#Self-loops valides
invalid_loops = self_loops[
    (self_loops["travel_time_sec"] != 0) |
    (self_loops["delta_lat"].abs() > 1e-6) |
    (self_loops["delta_lon"].abs() > 1e-6)
]

assert len(invalid_loops) == 0, f"Self-loops invalides : {len(invalid_loops)}"
print("✅ TEST SELF-LOOPS")


✅ TEST SELF-LOOPS


In [15]:
#travel_time_sec pour edges normaux
invalid_time = normal_edges[normal_edges["travel_time_sec"] <= 0]
assert len(invalid_time) == 0, f"Edges normaux invalides : {len(invalid_time)}"

print("✅ TEST travel_time_sec > 0")


✅ TEST travel_time_sec > 0


In [16]:
#Tests GÉOGRAPHIQUES
geo_errors = df[
    (df["from_lat"].abs() > 90) |
    (df["to_lat"].abs() > 90) |
    (df["from_lon"].abs() > 180) |
    (df["to_lon"].abs() > 180)
]

assert len(geo_errors) == 0, f"Coordonnées invalides : {len(geo_errors)}"
print("✅ TEST lat/lon")


✅ TEST lat/lon


In [17]:
#Doublons métier
dups = df.duplicated(
    subset=["trip_id","from_stop_id","to_stop_id","travel_time_sec"]
).sum()

assert dups == 0, f"Doublons détectés : {dups}"
print("✅ TEST doublons")


✅ TEST doublons


In [20]:
#Ratio self-loops
ratio = len(self_loops) / len(df)
assert ratio < 0.3, f"Trop de self-loops ({ratio:.2%})"

print(f"✅ TEST ratio self-loops ({ratio:.2%})")


✅ TEST ratio self-loops (21.62%)


In [ ]:
#Trips avec au moins un edge normal
trip_ok = (
    normal_edges.groupby("trip_id")
    .size()
)

bad_trips = df["trip_id"].nunique() - trip_ok.count()
assert bad_trips == 0, f"Trips sans edge normal : {bad_trips}"

print("✅ TEST trips valides")
